# Phase 1 : Business Understanding¶


## Problem Statement

While traditional decision trees and even ensemble methods such as Random Forests remain widely used in machine learning, they exhibit important limitations when dealing with high-dimensional datasets, noisy variables, or small node sizes during tree construction. In such situations, the splitting decisions become increasingly unstable and may approach random behavior, reducing the model’s predictive quality and its ability to correctly identify the most relevant variables. The RLT method was specifically designed to address these issues: it enhances split selection through a reinforcement learning mechanism that considers long term rewards, progressively removes uninformative variables through a variable muting procedure, and allows for more flexible linear-combination splits. By evaluating RLT across a diverse set of real world classification and regression tasks, the goal is to determine whether it provides improved stability, clearer interpretability, and superior predictive performance compared to classical tree-based methods.

### Business Objective 1 : 

The goal is to explore a new method "Reinforcement Learning Trees (RLT) " to improve predictive performance when data become large or complex.

### Business Objective 2 : 

We want to improve decision quality by reducing the influence of useless information: since datasets often contain noise, the aim is to build a system that focuses more on the information that truly matters.

### Business Objective 3 : 

The objective is to achieve an approach that remains stable, fast, and reliable result even when the structure of the data makes learning difficult for standard models.

### Data Science Objective 1 :

We will evaluate the behavior of RLT across different types of data .

### Data Science Objective 2 :

We want to analyze how RLT selects important information: since models sometimes rely on data that are not useful, the objective is to verify whether RLT can better identify meaningful indicators and filter out those that weaken the results.

### Data Science Objective 3 :

We will compare RLT with classical methods to move from a simple assumption to a precise evaluation of its performance. The aim is to determine whether it truly offers an improvement and to identify what could be enhanced if limitations appear.

# Phase 2 : Data Understanding

In [13]:
import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold


# -------------------------------------------------------
# 1. IMPORTATION AUTOMATIQUE DU DATASET
# -------------------------------------------------------
def load_dataset(path):
    if os.path.isdir(path):
        print("📁 Dossier détecté → Fusion des fichiers contenus.")
        files = os.listdir(path)
        dfs = []

        for f in files:
            full_path = os.path.join(path, f)
            if f.endswith(".csv"):
                dfs.append(pd.read_csv(full_path))
            elif f.endswith(".xlsx"):
                dfs.append(pd.read_excel(full_path, engine="openpyxl"))
            elif f.endswith(".xls"):
                dfs.append(pd.read_excel(full_path, engine="xlrd"))

        return pd.concat(dfs, ignore_index=True)

    if path.endswith(".csv"):
        return pd.read_csv(path)

    elif path.endswith(".xlsx"):
        return pd.read_excel(path, engine="openpyxl")

    elif path.endswith(".xls"):
        return pd.read_excel(path, engine="xlrd")

    elif path.endswith(".json"):
        return pd.read_json(path)

    else:
        raise ValueError("❌ Format non supporté (utilise CSV, Excel ou JSON).")





# -------------------------------------------------------
# 2. DATA UNDERSTANDING
# -------------------------------------------------------

def data_understanding(df):
    print("🔹 Shape:", df.shape)
    print("\n🔹 Les premières lignes:",df.head())
    print("\n🔹 Les dernières lignes:",df.tail())
    print("\n🔹 Info importants:",df.info)
    print("\n🔹 Colonnes:", df.columns.tolist())
    print("\n🔹 Types de variables:")
    print(df.dtypes)
    print("\n🔹 Valeurs manquantes par variable:")
    print(df.isnull().sum())
    print("\n🔹 Pourcentage des valeurs manquantes :")
    print(df.isnull().mean() * 100 )
    print("\n🔹 Nombre de valeurs uniques :")
    print(df.nunique())
   


    
    print("\n🔹 Détection des doublons :")
    print(df.duplicated().sum())
    print("\n🔹 Statistiques descriptives:")
    print(df.describe(include='all'))

    print("\n🔹 Détection des outliers (IQR) :")
    numeric_cols = df.select_dtypes(include=np.number).columns
    outlier_counts = {}
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        seuil_inf = Q1 - 1.5 * IQR
        seuil_sup = Q3 + 1.5 * IQR
        outliers = df[(df[col] < seuil_inf) | (df[col] > seuil_sup)]
        outlier_counts[col] = len(outliers)

    print("\n🔹 Nombre d'outliers par variable:")
    print(outlier_counts)

    return outlier_counts
# -------------------------------------------------------
# 3. DATA UNDERSTANDING
# -------------------------------------------------------

def data_preparation(df, target_column):

    # 0️⃣ Supprimer colonnes 100% vides
    # Supprimer les colonnes complètement vides
    df = df.dropna(axis=1, how='all')
    # 1️⃣ Séparer X et y
    X = df.drop(columns=[target_column])
    y = df[target_column]

    # 2️⃣ Colonnes numériques / catégorielles
    print("NUMERIC VARIABLES:")
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    print(numeric_cols) 
    print("CATEGORICA VARIABLES:")
    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    print(categorical_cols) 
    # 3️⃣ Pipeline numérique
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('variance', VarianceThreshold(threshold=0.0)),
        ('scaler', StandardScaler())
    ])

    # 4️⃣ Pipeline catégoriel
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    # 5️⃣ Transformer
    preprocessor = ColumnTransformer(transformers=[
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ])

    # 6️⃣ Transformation
    X_prepared = preprocessor.fit_transform(X)

    return X_prepared, y, preprocessor



def full_pipeline(path, target_column):
    print("\n========== 📥 IMPORTATION DATA ==========")
    df = load_dataset(path)

    print("\n========== 🧠 DATA UNDERSTANDING ==========")
    data_understanding(df)
    print("\n========== ⚙️ DATA PREPARATION ==========")
    data_preparation(df, target_column)

   

